# Rolling linear regression

Slope, R-squared and the residual produced by fitting a straight line to a trailing window of log mid price.

**This notebook contains no calculations.** Everything is computed by
`xauusd_quant.features` and `xauusd_quant.research` and read back from
`results/regression_research/<timeframe>/window_<N>/`. Generate those first:

```
xq regression-research --timeframe 1h --window 128
xq regression-research --timeframe 5m --all-windows
xq regression-research --all
```

No trading rule is defined or evaluated anywhere in this notebook. No
transaction costs are applied. Nothing here is a signal.

In [ ]:
import sys
from pathlib import Path

# Work against the repository's own source tree, not an installed copy.
ROOT = Path.cwd().parent if Path.cwd().name == "research" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import polars as pl

from xauusd_quant import load_config
from xauusd_quant.features import load_regression_config

config = load_config(ROOT / "config" / "data.yaml")
regression = load_regression_config(ROOT / "config" / "regression.yaml")

TIMEFRAME = "1h"          # change and re-run
WINDOW = 128              # change and re-run

print(f"instrument : {config.instrument}")
print(f"timezone   : {config.timezone.describe()}")
print(f"transform  : {regression.regression.price_transform} of "
      f"{regression.regression.price_source} {regression.regression.price_column}")
print(f"windows    : {list(regression.regression.windows)}")
print(f"fingerprint: {regression.fingerprint()}")

In [ ]:
import json

results = regression.results_dir(TIMEFRAME, WINDOW)
if not results.exists():
    raise SystemExit(
        f"No results for {TIMEFRAME} window={WINDOW}. Run:  "
        f"xq regression-research --timeframe {TIMEFRAME} --window {WINDOW}"
    )

def table(name):
    """Load a previously-generated regression table."""
    path = results / f"{name}.parquet"
    return pl.read_parquet(path) if path.exists() else None

summary = json.loads((results / "summary.json").read_text(encoding="utf-8"))
provenance = summary["provenance"]
model = summary.get("model_summary", {})

print(f"generated  : {provenance['generated_utc']}")
print(f"window     : {provenance['regression_window']} bars on {provenance['timeframe']}")
print(f"valid fits : {provenance['observations']:,}")
print(f"span       : {provenance['requested_start']} -> {provenance['requested_end']}")
print(f"git        : {provenance['git']}")
for warning in summary.get("warnings", []):
    print(f"WARNING: {warning}")

## What is being fitted

For each bar `t`, ordinary least squares is fitted to the window **ending at
`t`**:

```
y_i = alpha_t + beta_t * x_i + eps_i,   i = 0 .. N-1,   x_i = i
```

`y` is the log of the mid close, `x` is the bar's position inside the window,
and the last point of the window is the current bar. The residual reported at
`t` is the residual of that last point, `eps_{N-1}`.

Because `x` is a fixed integer ramp, `x_bar = (N-1)/2` and
`Sxx = N(N^2-1)/12` are constants: the fit is a couple of dot products per
bar, which is why the whole grid is cheap.

Two facts follow from the construction alone and are worth holding on to:

1. The residual has mean zero **within each window** by definition. That is
   arithmetic, not evidence of anything.
2. Consecutive windows share `N-1` of their `N` bars, so consecutive residuals
   are fitted against almost the same line. High residual autocorrelation is
   guaranteed before any market behaviour enters.

In [ ]:
diagnostics = summary["feature_diagnostics"]
for key, value in diagnostics.items():
    print(f"{key:<28} {value}")

## Slope

`slope` is the per-bar change in log price implied by the local trend, and
`mean_pct_per_bar` is the same thing as a percentage. Neither is annualised:
that would need an explicit bars-per-year assumption, which this project does
not make silently.

In [ ]:
table('slope_statistics')

## R-squared, and what the residual does inside each quartile

A high R-squared means the window was well described by a straight line; a low
one means the local path was not trend-like. Whether the residual behaves
differently in the two cases is a real question rather than an assumption, so
the residual standard deviation and the extreme frequency are reported per
quartile of R-squared.

In [ ]:
table('r_squared_statistics')

## Residual distribution, and the two Z-scores

Two standardisations are kept in **separate columns** and never mixed:

- `residual_zscore_fit` divides the residual by the standard error of its own
  fit. It says how unusual the last point is relative to the line that was
  fitted through it.
- `residual_zscore_rolling` subtracts a rolling mean of the residual series and
  divides by its rolling standard deviation. It says how unusual the residual
  is relative to recent residuals.

They answer different questions and generally disagree. Reporting one as if it
were the other is the mistake this split exists to prevent.

In [ ]:
table('residual_distribution')

## Residual autocorrelation

Read the three series together:

- `residual` - high lag-1 autocorrelation is **mechanical**, see above.
- `residual_diff` - the differenced series, which removes most of the overlap
  artefact and is the more informative object.
- `abs_residual` - volatility clustering in the residual.

A Ljung-Box rejection on `residual` therefore tells you almost nothing. On
`residual_diff` it is worth looking at.

In [ ]:
table('residual_acf')

In [ ]:
table('residual_ljung_box')

## Across windows

`model_comparison` is written when more than one model is generated in a single
run. **There is deliberately no "best" window here.** Different windows describe
different horizons; picking the one with the most attractive statistic is
parameter selection on the full sample, which is exactly what Prompt #3
forbids at this stage.

In [ ]:
comparison = regression.results_path / "model_comparison.parquet"
if comparison.exists():
    frame = pl.read_parquet(comparison)
    print(f"{frame.height} rows across "
          f"{frame['timeframe'].n_unique()} timeframe(s) and "
          f"{frame['regression_window'].n_unique()} window(s)")
    frame
else:
    print("Not generated yet. Run:  xq regression-research --all")

## Figures

In [ ]:
plots = sorted((results / "plots").glob("*.png")) if (results / "plots").exists() else []
for path in plots:
    print(path.name)
print(f"\n{len(plots)} figure(s) in {results / 'plots'}")

In [ ]:
from IPython.display import Image, display

for path in plots:
    print(path.name)
    display(Image(filename=str(path)))